In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestRegressor
from sklearn.inspection import permutation_importance
from sklearn.neural_network import MLPRegressor
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_squared_error, r2_score

# -------------------- Load Data --------------------
df = pd.read_csv("synthetic_battery_data.csv")

features = ['electrode_thickness', 'separator_thickness', 'elastic_modulus',
            'diffusion_coefficient', 'applied_current', 'temperature', 'external_pressure']
targets = ['voltage_drop', 'max_stress', 'lithium_concentration']

X = df[features]
y = df[targets]

# Split data
X_train, X_val, y_train, y_val = train_test_split(X, y, test_size=0.2, random_state=42)

# -------------------- Random Forest Feature Importance --------------------

feature_ranking = {}
for t in targets:
    rf_model = RandomForestRegressor(n_estimators=300, random_state=42)
    rf_model.fit(X_train, y_train[t])  # single target
    importances = rf_model.feature_importances_
    indices = permutation_importance(rf_model, X_val, y_val[t], n_repeats=10, random_state=42, scoring='r2')
    feature_ranking[t] = {
        "rf_model_importance": dict(zip(features, importances)),
        "perm_importance": dict(zip(features, indices.importances_mean))
    }


#rf_model = RandomForestRegressor(n_estimators=100, random_state=42)
#rf_model.fit(X_train, y_train)

#importances = rf_model.feature_importances_
#indices = np.argsort(importances)[::-1]

plt.figure(figsize=(8, 5))
plt.title("Feature Importances (Random Forest)")
plt.bar(range(len(features)), importances[indices], align="center")
plt.xticks(range(len(features)), [features[i] for i in indices], rotation=45)
plt.tight_layout()
plt.show()

# -------------------- Neural Network with MLPRegressor --------------------
# Scale features and targets
scaler_X = StandardScaler()
X_train_scaled = scaler_X.fit_transform(X_train)
X_val_scaled = scaler_X.transform(X_val)

scaler_y = StandardScaler()
y_train_scaled = scaler_y.fit_transform(y_train)
y_val_scaled = scaler_y.transform(y_val)

# Define MLPRegressor
mlp = MLPRegressor(hidden_layer_sizes=(32, 16), activation='relu',
                   solver='adam', max_iter=1000, random_state=42)

# Train model
mlp.fit(X_train_scaled, y_train_scaled)

# Predict and evaluate
y_pred_scaled = mlp.predict(X_val_scaled)
y_pred = scaler_y.inverse_transform(y_pred_scaled)

mse = mean_squared_error(y_val, y_pred)
r2 = r2_score(y_val, y_pred)

print("MLPRegressor Evaluation:")
print(f"Mean Squared Error: {mse:.4f}")
print(f"R² Score: {r2:.4f}")

# -------------------- Physics-Inspired Penalty Check --------------------
penalty_count = 0
for i, pred in enumerate(y_pred):
    voltage, stress, concentration = pred
    if voltage > 5.0 or stress < 0:
        penalty_count += 1
        print(f"Warning: Sample {i} violates physics constraints -> Voltage: {voltage:.2f}, Stress: {stress:.2f}")

print(f"Total violations: {penalty_count} out of {len(y_pred)}")

# -------------------- Optional: Plot Loss Curve --------------------
plt.figure(figsize=(8, 5))
plt.plot(mlp.loss_curve_)
plt.title("Training Loss Curve")
plt.xlabel("Iterations")
plt.ylabel("Loss")
plt.show()

#print(mlp.coefs_)
print(mlp.intercepts_)

Note: you may need to restart the kernel to use updated packages.


NameError: name 'rf' is not defined